# 🚀 Zoodex Cloud Backend su Google Colab Pro

Questo notebook esegue il backend FastAPI di Zoodex con accelerazione GPU (NVIDIA A100 / L4 / T4) e genera un **URL HTTPS pubblico permanente tramite Cloudflare Tunnel**.

### Istruzioni rapide:
1. In alto su Colab Pro: seleziona **Runtime** -> **Modifica tipo di runtime** -> **GPU** (es. A100, L4 o T4) e **RAM elevata**.
2. Clicca su **Runtime** -> **Esegui tutto**.
3. Al termine dell'esecuzione della cella 3, vedrai stampato a caratteri cubitali il link HTTPS pubblico (es. `https://xxxx.trycloudflare.com`).
4. Incolla questo link nelle impostazioni dell'app Zoodex sul telefono (tasto ⚙️ in alto a destra).
5. Puoi interrompere il server in qualsiasi momento premendo il tasto **Stop ⏹️** sulla cella 3.

In [ ]:
# 1. Clona il repository Zoodex ed installa le dipendenze
!rm -rf zoodex
!git clone https://github.com/Emamasi-uni/zoodex.git
%cd zoodex/backend
!pip install --upgrade pip -q
!pip install -r requirements.txt -q

In [ ]:
# 2. Scarica Cloudflare Tunnel (gratuito, sicuro e senza account richiesto)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

In [ ]:
# 3. Avvia FastAPI e genera l'URL Cloud pubblico
# Termina eventuali processi precedenti per liberare la porta 8000
!pkill -9 -f uvicorn 2>/dev/null || true
!pkill -9 -f cloudflared 2>/dev/null || true
!git pull origin main -q

import subprocess
import time
import re

time.sleep(1)
# Avvia uvicorn in background sulla porta 8000
api_process = subprocess.Popen(["uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"])
time.sleep(3)

# Avvia il tunnel Cloudflare verso la porta 8000
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print("Creazione del tunnel Cloudflare in corso...")
cloud_url = None
for line in tunnel_proc.stderr:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        cloud_url = match.group(0)
        print("\n" + "="*72)
        print("🎉 BACKEND ZOODEX ATTIVO CON ACCELERAZIONE GPU SU COLAB PRO!")
        print(f"🔗 URL CLOUD DA INCOLLARE NELL'APP:  {cloud_url}")
        print(f"🧪 Health Check: {cloud_url}/health")
        print(f"ℹ️ Modello attivo: {cloud_url}/api/v1/model-info")
        print("="*72 + "\n")
        print("Server in ascolto... (premi Stop ⏹️ sulla cella quando vuoi arrestarlo)")
        break

try:
    tunnel_proc.wait()
except KeyboardInterrupt:
    api_process.terminate()
    tunnel_proc.terminate()
    print("Server e tunnel arrestati correttamente.")